# Shi hypotheses — why did two similarly-active storms produce order-of-magnitude different GIC at the same substation?

**Companion notebook** to the in-preparation paper. Reproduces every figure inline via the scripts in `../scripts/`. `Restart Kernel & Run All` regenerates every figure from scratch.

## The three hypotheses

1. **H1 (spatial-field)** — the *local* geoelectric field at the target substation is the same between the two storms, but the *regional* field structure differs. GIC is set by the whole network's current divider, so a purely-spatial change in the field elsewhere in the grid can move GIC at the observation point by an order of magnitude while the local driving field stays fixed.
2. **H2 (mitigation)** — the operator raised neutral-side grounding at the target substation between the storms.
3. **H3 (reconfiguration)** — network topology changed (a line was out of service, a new tie was in service), redistributing where GIC concentrates.

Each hypothesis is a testable, isolated claim. The scripts quantify how large an effect each mechanism can produce on the Horton (2012) 21-bus EPRI test network under a synthetic 1 V/km eastward baseline field.

Target substation across all three: `dc_sub6`.

## Network setup

Horton EPRI21 as delivered: 8 substations, 15 transmission lines (500 kV + 345 kV tiers), 22 GSU / low-side branches. Two substations (`dc_sub1`, `dc_sub7`) have NaN coordinates in the shipped MATPOWER file so are omitted from the map — see issue #22 for the tracking follow-up.

In [ ]:
from IPython.display import Image, display

%run ../scripts/plot_network_map_setup.py
display(Image("../figures/fig3_network_map.png"))

## H1 — spatial-field sensitivity

Sweep `dE_x/dx` across ±0.02 V/km per km while holding `E_x(target) = 1.0 V/km` fixed. The gradient field is constructed as `E_x(x) = E_local - dE_x/dx · (x - x_target)` in the local equirectangular frame so the field at the target is invariant across every run — only its spatial variation elsewhere changes.

**H1a** — GIC at the target vs the gradient. The `±1 order of magnitude` band shows how far a purely-spatial change can push the target current. **H1b** — the same story on the network map: uniform (left) vs one representative gradient (right); target ringed in blue; per-substation `|GIC|` labelled.

In [ ]:
%run ../scripts/hypothesis1_spatial_field.py
display(Image("../figures/h1a_sensitivity_curve.png"))
display(Image("../figures/h1b_redistribution_map.png"))

## H2 — mitigation

Hold the field at the uniform 1 V/km baseline. Sweep the grounding-resistance multiplier at the target from 1× (as-delivered) to 100× on a log grid, plus a `1e12 Ω` "full blocker" limiting case. Twin axes: **red** = target `|GIC|` (local protection metric); **blue** = network Σ`|GIC|` (network cost metric).

Both fall together as `R_gnd` rises — local protection *and* whole-network current both drop, because the target substation is a significant sink under the uniform field. Cheap grounding upgrades at the target buy real reduction; the returns diminish sharply above ~10× baseline.

In [ ]:
%run ../scripts/hypothesis2_mitigation.py
display(Image("../figures/h2a_local_vs_global.png"))

## H3 — reconfiguration

Baseline field fixed. Open every transmission line one at a time, and add one hypothetical long-distance tie between `dc_sub2` (NW-most) and `dc_sub8` (E-most). Ranked bar chart of `Δ|GIC|` at the target.

**Blue bars** (negative Δ): outages that *lowered* GIC at the target — those lines were feeders into the target region. **Red bars** (positive Δ): outages that *raised* it — those lines were diverting current elsewhere. Biggest positive: opening `dc_br6` (from target itself out to `dc_bus11`) → +129 A. Biggest negative: opening `dc_br7` → -53 A. The added tie has near-zero target impact but cuts network Σ`|GIC|` by 8%.

In [ ]:
%run ../scripts/hypothesis3_reconfiguration.py
display(Image("../figures/h3_reconfiguration_bars.png"))

## Taking stock

All three mechanisms produce order-of-magnitude-scale effects at the target under physically-plausible perturbations:

| Hypothesis | Range at target `dc_sub6` (baseline = 354 A) | Notes |
|---|---|---|
| **H1** spatial-field | -705 A to +1413 A across `dE/dx ∈ ±0.02 V/km per km` | ~4× dynamic range in either direction |
| **H2** mitigation | 354 A → 14 A (100× R_gnd) → 0 A (full blocker) | monotone, diminishing returns |
| **H3** reconfiguration | -53 A to +129 A across 15 single-line outages | biggest hit = opening the target's own outbound line |

Any of the three, alone, could explain a factor-10 storm-to-storm difference at a single observation point. The paper argues that between two real storms all three probably act simultaneously and the interesting question is not "which one" but how to attribute the observed change to each contribution.

## What this notebook does *not* do

- **No real storm data.** The point is mechanism-quantification with synthetic fields. Real spatially-resolved storm E-fields would come from a SECS driver (Weygand et al. 2011), a v0.5 GeoPulse item.
- **No publication-quality figures.** Everything is draft-quality matplotlib. When the paper is being submitted, `geopulse.viz.presets.apply_preset('sw_2col')` (or `jgr_2col`, etc.) plus `save_figure(...)` drop in via a two-line change per script.
- **No transformer thermal / harmonics.** Deferred by design — separate downstream questions once the three mechanism-level claims are settled.